# LOAD FROM S3

In [15]:
import requests
from pathlib import Path

BASE_DIR = Path().resolve().parents[2]
output_path = BASE_DIR / "data" / "raw" / "flights_dataset.zip"

url = "https://ppml2026.s3.eu-north-1.amazonaws.com/datasets/SignofFlightsDataset_20260407_222001.zip"

response = requests.get(url)

with open(output_path, "wb") as f:
    f.write(response.content)

print("Download OK : ", output_path)

Download OK :  /Users/manjakaranjatoson/Desktop/A_I/JEDHA/PROJECTS/CDSD/PPML/data/raw/flights_dataset.zip


The zip file is corrupted so we manually loaded the csv.

# EDA

In [16]:
import pandas as pd

df_train_philippe = pd.read_csv("/Users/manjakaranjatoson/Desktop/A_I/JEDHA/PROJECTS/CDSD/PPML/data/raw/00_philippe_dataset_train.csv")
df_copy = df_train_philippe.copy()

/var/folders/p7/7w66qcfn4yj82k3sbs5fp74h0000gn/T/ipykernel_60786/4201604161.py:3: DtypeWarning: Columns (64,65,67,75,79,81,83,85) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train_philippe = pd.read_csv("/Users/manjakaranjatoson/Desktop/A_I/JEDHA/PROJECTS/CDSD/PPML/data/raw/00_philippe_dataset_train.csv")


In [17]:
df_train_philippe.head()

,flight_date,movement_date,flight_number,airline,airport_origin,airport_destination,terminal_departure,terminal_arrival,scheduled_departure,scheduled_arrival,...,GREVE_ORLY,LABEL_ORLY,nombre_departs_source,nombre_arrivees_source,somme_depart_arrivee_source,congestion_source,nombre_departs_destination,nombre_arrivees_destination,somme_depart_arrivee_destination,congestion_destination
0,DATE_GENERATION,2026-04-08 05:21:09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-10-11,2025-10-11,5O 701,ASL Airlines France,CDG,MRS,NaN,NaN,2025-10-11 01:39+02:00,2025-10-11 02:34+02:00,...,Non,NaN,616.0,620.0,1236.0,0.0,119.0,113.0,232.0,0.0
2,2025-10-11,2025-10-11,AF 6001,Air France,MRS,ORY,1,2,2025-10-11 06:20+02:00,2025-10-11 07:45+02:00,...,Non,NaN,119.0,113.0,232.0,0.0,299.0,299.0,598.0,0.0
3,2025-10-11,2025-10-11,AF 6004,Air France,ORY,MRS,3,1,2025-10-11 09:00+02:00,2025-10-11 10:20+02:00,...,Non,NaN,299.0,299.0,598.0,0.0,119.0,113.0,232.0,0.0
4,2025-10-11,2025-10-11,AF 6009,Air France,MRS,ORY,1,2,2025-10-11 15:40+02:00,2025-10-11 17:05+02:00,...,Non,NaN,119.0,113.0,232.0,0.0,299.0,299.0,598.0,0.0


In [18]:
print(f"Dataset shape :", df_train_philippe.shape)
print()
print(f"Dataset columns and type :", df_train_philippe.info())
print()
print(f"Dataset NaN values :", df_train_philippe.isna().sum().sort_values(ascending=False))
print()


Dataset shape : (88098, 94)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88098 entries, 0 to 88097
Data columns (total 94 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   flight_date                       88098 non-null  object 
 1   movement_date                     88098 non-null  object 
 2   flight_number                     88097 non-null  object 
 3   airline                           88097 non-null  object 
 4   airport_origin                    88097 non-null  object 
 5   airport_destination               88097 non-null  object 
 6   terminal_departure                74281 non-null  object 
 7   terminal_arrival                  75174 non-null  object 
 8   scheduled_departure               87082 non-null  object 
 9   scheduled_arrival                 86798 non-null  object 
 10  estimated_departure               87082 non-null  object 
 11  estimated_arrival                 8679

## Data cleaning
* Suppressing first line
* Filtering to domestic flights
* Handling Duplicates
* Handling NaN values

**For the classification dataset**
* Add a col Retard

**For the regression dataset**
* Feature engineering

### Supressing first line

In [19]:
print(df_train_philippe.iloc[0])
df_train_philippe = df_train_philippe.iloc[1:].reset_index(drop=True)

display(df_train_philippe.head())

flight_date                             DATE_GENERATION
movement_date                       2026-04-08 05:21:09
flight_number                                       NaN
airline                                             NaN
airport_origin                                      NaN
                                           ...         
congestion_source                                   NaN
nombre_departs_destination                          NaN
nombre_arrivees_destination                         NaN
somme_depart_arrivee_destination                    NaN
congestion_destination                              NaN
Name: 0, Length: 94, dtype: object


,flight_date,movement_date,flight_number,airline,airport_origin,airport_destination,terminal_departure,terminal_arrival,scheduled_departure,scheduled_arrival,...,GREVE_ORLY,LABEL_ORLY,nombre_departs_source,nombre_arrivees_source,somme_depart_arrivee_source,congestion_source,nombre_departs_destination,nombre_arrivees_destination,somme_depart_arrivee_destination,congestion_destination
0,2025-10-11,2025-10-11,5O 701,ASL Airlines France,CDG,MRS,NaN,NaN,2025-10-11 01:39+02:00,2025-10-11 02:34+02:00,...,Non,NaN,616.0,620.0,1236.0,0.0,119.0,113.0,232.0,0.0
1,2025-10-11,2025-10-11,AF 6001,Air France,MRS,ORY,1,2,2025-10-11 06:20+02:00,2025-10-11 07:45+02:00,...,Non,NaN,119.0,113.0,232.0,0.0,299.0,299.0,598.0,0.0
2,2025-10-11,2025-10-11,AF 6004,Air France,ORY,MRS,3,1,2025-10-11 09:00+02:00,2025-10-11 10:20+02:00,...,Non,NaN,299.0,299.0,598.0,0.0,119.0,113.0,232.0,0.0
3,2025-10-11,2025-10-11,AF 6009,Air France,MRS,ORY,1,2,2025-10-11 15:40+02:00,2025-10-11 17:05+02:00,...,Non,NaN,119.0,113.0,232.0,0.0,299.0,299.0,598.0,0.0
4,2025-10-11,2025-10-11,AF 6100,Air France,ORY,TLS,3,NaN,2025-10-11 07:00+02:00,2025-10-11 08:15+02:00,...,Non,NaN,299.0,299.0,598.0,0.0,78.0,73.0,151.0,0.0


### Filtering to domestic flights

In [39]:
french_airports = ["CDG", "ORY", "LYS", "MRS", "NCE", "TLS"]

df_train_philippe = df_train_philippe[
    (df_train_philippe["airport_origin"].isin(french_airports)) &
    (df_train_philippe["airport_destination"].isin(french_airports))
].copy()

display(df_train_philippe["airport_origin"].unique())
display(df_train_philippe["airport_destination"].unique())

array(['CDG', 'MRS', 'ORY', 'TLS', 'NCE', 'LYS'], dtype=object)

array(['MRS', 'ORY', 'TLS', 'NCE', 'CDG', 'LYS'], dtype=object)

### Handling duplicates

In [40]:
df_train_philippe[df_train_philippe.duplicated(subset=["flight_number", "flight_date"], keep=False)]

,flight_date,movement_date,flight_number,airline,airport_origin,airport_destination,terminal_departure,terminal_arrival,scheduled_departure,scheduled_arrival,...,GREVE_ORLY,LABEL_ORLY,nombre_departs_source,nombre_arrivees_source,somme_depart_arrivee_source,congestion_source,nombre_departs_destination,nombre_arrivees_destination,somme_depart_arrivee_destination,congestion_destination
207,2025-10-12,2025-10-12,AF 7303,Air France,NCE,CDG,2,NaN,2025-10-12 11:05+02:00,NaN,...,Non,NaN,177.0,180.0,357.0,1.0,643.0,631.0,1274.0,0.0
208,2025-10-12,2025-10-12,AF 7303,Air France,NCE,CDG,NaN,2F,NaN,2025-10-12 12:40+02:00,...,Non,NaN,177.0,180.0,357.0,1.0,643.0,631.0,1274.0,0.0
962,2025-10-15,2025-10-15,AF 7419,Air France,TLS,CDG,NaN,NaN,2025-10-15 07:30+02:00,NaN,...,Non,NaN,99.0,96.0,195.0,0.0,624.0,565.0,1189.0,0.0
963,2025-10-15,2025-10-15,AF 7419,Air France,TLS,CDG,NaN,2F,NaN,2025-10-15 09:00+02:00,...,Non,NaN,99.0,96.0,195.0,0.0,624.0,565.0,1189.0,0.0
1313,2025-10-16,2025-10-16,WT 6RS,Swiftair,MRS,CDG,NaN,NaN,2025-10-16 22:18+02:00,NaN,...,Non,NaN,139.0,145.0,284.0,1.0,637.0,645.0,1282.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87888,2026-04-08,2026-04-08,MU 1512,China Eastern,LYS,CDG,1,2F,2026-04-08 14:40+02:00,2026-04-08 15:50+02:00,...,Non,NaN,230.0,235.0,465.0,0.0,2443.0,2467.0,4910.0,1.0
87896,2026-04-08,2026-04-08,MU 1531,China Eastern,CDG,MRS,2F,1,2026-04-08 09:55+02:00,2026-04-08 11:15+02:00,...,Non,NaN,2443.0,2467.0,4910.0,1.0,126.0,125.0,251.0,1.0
87897,2026-04-08,2026-04-08,MU 1531,China Eastern,CDG,MRS,2F,1,2026-04-08 21:25+02:00,2026-04-08 22:50+02:00,...,Non,NaN,2443.0,2467.0,4910.0,1.0,126.0,125.0,251.0,1.0
87907,2026-04-08,2026-04-08,MU 1768,China Eastern,LYS,CDG,1,NaN,2026-04-08 10:20+02:00,NaN,...,Non,NaN,230.0,235.0,465.0,0.0,2443.0,2467.0,4910.0,1.0


Nous avons identifié que les doublons apparents ne correspondent pas à des escales, mais à deux lignes complémentaires décrivant un même vol : l’une pour le départ et l’autre pour l’arrivée. Il ne faut donc pas supprimer ces lignes ni les traiter séparément, car cela entraînerait une perte d’information et un biais dans le modèle. La bonne approche consiste à regrouper ces lignes pour reconstruire un vol complet en une seule observation, en combinant les informations de départ et d’arrivée (par exemple via un groupby et des agrégations adaptées).


In [41]:
dup_mask = df_train_philippe.duplicated(
    subset=["flight_number", "flight_date",  "airport_origin", "airport_destination"],
    keep=False
)
df_dups = df_train_philippe[dup_mask]
df_clean = df_train_philippe[~dup_mask]

In [42]:
df_dups["movement_type"].value_counts()

movement_type
arrival      690
departure    676
Name: count, dtype: int64

In [49]:
import pandas as pd

# --- 1) copie
df = df_dups.copy()

# on garde l'ordre exact des colonnes d'origine
original_cols = df.columns.tolist()

# --- 2) conversion datetime
datetime_cols = [
    "flight_date",
    "movement_date",
    "scheduled_departure",
    "scheduled_arrival",
    "estimated_departure",
    "estimated_arrival",
    "actual_departure",
    "actual_arrival",
    "actual_source_departure",
    "actual_source_arrival",
]

for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

# --- 3) séparation départ / arrivée
df_dep = df[df["movement_type"] == "departure"].copy()
df_arr = df[df["movement_type"] == "arrival"].copy()

# --- 4) clé logique
keys = ["flight_number", "flight_date", "airport_origin", "airport_destination"]

# --- 5) tri
df_dep = df_dep.sort_values(keys + ["scheduled_departure", "estimated_departure", "actual_departure"])
df_arr = df_arr.sort_values(keys + ["scheduled_arrival", "estimated_arrival", "actual_arrival"])

# --- 6) occurrence pour distinguer les multi-vols même jour
df_dep["occurrence"] = df_dep.groupby(keys).cumcount()
df_arr["occurrence"] = df_arr.groupby(keys).cumcount()

# --- 7) merge avec TOUTES les colonnes
df_merge_raw = df_dep.merge(
    df_arr,
    on=keys + ["occurrence"],
    how="left",
    suffixes=("_dep_side", "_arr_side"),
    validate="1:1"
)

# --- 8) reconstruction des colonnes d'origine
df_merged = pd.DataFrame()

for col in original_cols:
    dep_col = f"{col}_dep_side"
    arr_col = f"{col}_arr_side"
    
    if dep_col in df_merge_raw.columns and arr_col in df_merge_raw.columns:
        # on prend d'abord la valeur côté départ, sinon côté arrivée
        df_merged[col] = df_merge_raw[dep_col].combine_first(df_merge_raw[arr_col])
    elif dep_col in df_merge_raw.columns:
        df_merged[col] = df_merge_raw[dep_col]
    elif arr_col in df_merge_raw.columns:
        df_merged[col] = df_merge_raw[arr_col]
    elif col in df_merge_raw.columns:
        # cas des colonnes de jointure
        df_merged[col] = df_merge_raw[col]

# on remet exactement l'ordre d'origine
df_merged = df_merged[original_cols].reset_index(drop=True)

print(df_dep.shape)
print(df_arr.shape)
print(df_merged.shape)
df_merged.head()

(676, 95)
(690, 95)
(676, 94)


,flight_date,movement_date,flight_number,airline,airport_origin,airport_destination,terminal_departure,terminal_arrival,scheduled_departure,scheduled_arrival,...,GREVE_ORLY,LABEL_ORLY,nombre_departs_source,nombre_arrivees_source,somme_depart_arrivee_source,congestion_source,nombre_departs_destination,nombre_arrivees_destination,somme_depart_arrivee_destination,congestion_destination
0,2025-11-23 00:00:00+00:00,2025-11-23 00:00:00+00:00,AF 6110,Air France,ORY,TLS,3,NaN,2025-11-23 11:00:00+00:00,2025-11-23 12:15:00+00:00,...,Non,NaN,437.0,447.0,884.0,0.0,77.0,79.0,156.0,0.0
1,2025-11-23 00:00:00+00:00,2025-11-23 00:00:00+00:00,AF 6116,Air France,ORY,TLS,3,NaN,2025-11-23 15:10:00+00:00,2025-11-23 16:20:00+00:00,...,Non,NaN,437.0,447.0,884.0,0.0,77.0,79.0,156.0,0.0
2,2025-11-15 00:00:00+00:00,2025-11-15 00:00:00+00:00,AF 6201,Air France,NCE,ORY,2,2,2025-11-15 05:45:00+00:00,2025-11-15 07:15:00+00:00,...,Non,NaN,95.0,93.0,188.0,0.0,235.0,237.0,472.0,0.0
3,2025-10-12 00:00:00+00:00,2025-10-12 00:00:00+00:00,AF 7303,Air France,NCE,CDG,2,2F,2025-10-12 09:05:00+00:00,2025-10-12 10:40:00+00:00,...,Non,NaN,177.0,180.0,357.0,1.0,643.0,631.0,1274.0,0.0
4,2025-10-23 00:00:00+00:00,2025-10-23 00:00:00+00:00,AF 7323,Air France,NCE,CDG,2,2F,2025-10-23 11:55:00+00:00,2025-10-23 13:30:00+00:00,...,Non,NaN,174.0,175.0,349.0,1.0,673.0,637.0,1310.0,0.0


In [50]:
df_merged[df_merged.duplicated(
    subset=[
        "flight_number",
        "flight_date",
        "airport_origin",
        "airport_destination",
        "scheduled_departure"
    ],
    keep=False
)]

,flight_date,movement_date,flight_number,airline,airport_origin,airport_destination,terminal_departure,terminal_arrival,scheduled_departure,scheduled_arrival,...,GREVE_ORLY,LABEL_ORLY,nombre_departs_source,nombre_arrivees_source,somme_depart_arrivee_source,congestion_source,nombre_departs_destination,nombre_arrivees_destination,somme_depart_arrivee_destination,congestion_destination


Good ! Nous pouvons supposer que les duplicates (arrivés et départ) ont été correctement mergés. 

In [58]:
# Concat df_merged with df_clean cf. See cell 41
df = pd.concat([df_clean, df_merged], ignore_index=True)

print(df.shape)
print(df_clean.shape)
print(df_merged.shape)

(87407, 94)
(86731, 94)
(676, 94)


#### Handling NaN Values

In [61]:
(df.isna().mean() * 100).sort_values(ascending=False)

LABEL_TOULOUSE    100.0
visibility_arr    100.0
cloud_base_dep    100.0
cloud_base_arr    100.0
visibility_dep    100.0
                  ...  
Vacances NCE        0.0
GREVE_TOULOUSE      0.0
Vacances TLS        0.0
GREVE_LYON          0.0
flight_date         0.0
Length: 95, dtype: float64

In [76]:
# % de NaN
na_pct = df.isna().mean() * 100

# seuil (tu peux ajuster)
threshold = 65

# colonnes à drop
cols_to_drop = na_pct[na_pct > threshold].index.tolist()

print("Colonnes supprimées :", cols_to_drop)

# drop
df = df.drop(columns=cols_to_drop)

# check
print(df.shape)

Colonnes supprimées : ['Vacances Scolaires', 'Label des Vacances']
(87407, 87)


### Classification dataset :

#### Adding a column Retard : 1 | 0 si departure_delay_min >= 15 min

In [60]:
df["Retard"] = df["departure_delay_min"].apply(lambda x: 1 if x >= 15 else 0)

# On met la col Retard à côté de departure_delay_min
col = df.pop("Retard")
df.insert(
    df.columns.get_loc("departure_delay_min") + 1,
    "Retard",
    col
)

df.head()

,flight_date,movement_date,flight_number,airline,airport_origin,airport_destination,terminal_departure,terminal_arrival,scheduled_departure,scheduled_arrival,...,GREVE_ORLY,LABEL_ORLY,nombre_departs_source,nombre_arrivees_source,somme_depart_arrivee_source,congestion_source,nombre_departs_destination,nombre_arrivees_destination,somme_depart_arrivee_destination,congestion_destination
0,2025-10-11,2025-10-11,5O 701,ASL Airlines France,CDG,MRS,NaN,NaN,2025-10-11 01:39+02:00,2025-10-11 02:34+02:00,...,Non,NaN,616.0,620.0,1236.0,0.0,119.0,113.0,232.0,0.0
1,2025-10-11,2025-10-11,AF 6001,Air France,MRS,ORY,1,2,2025-10-11 06:20+02:00,2025-10-11 07:45+02:00,...,Non,NaN,119.0,113.0,232.0,0.0,299.0,299.0,598.0,0.0
2,2025-10-11,2025-10-11,AF 6004,Air France,ORY,MRS,3,1,2025-10-11 09:00+02:00,2025-10-11 10:20+02:00,...,Non,NaN,299.0,299.0,598.0,0.0,119.0,113.0,232.0,0.0
3,2025-10-11,2025-10-11,AF 6009,Air France,MRS,ORY,1,2,2025-10-11 15:40+02:00,2025-10-11 17:05+02:00,...,Non,NaN,119.0,113.0,232.0,0.0,299.0,299.0,598.0,0.0
4,2025-10-11,2025-10-11,AF 6100,Air France,ORY,TLS,3,NaN,2025-10-11 07:00+02:00,2025-10-11 08:15+02:00,...,Non,NaN,299.0,299.0,598.0,0.0,78.0,73.0,151.0,0.0


In [77]:
df["movement_type"].value_counts()

movement_type
arrival      43897
departure    43510
Name: count, dtype: int64